# Phase 3 - Hoang Grounded Output Research

Notebook nay tap trung vao phan cuoi cua Vinh Hoang:
1. Load phase 1 artifact
2. Consume San phase 2 contract (or local retrieval/mock fallback)
3. Generate grounded answer and export `phase3_hoang_grounded_answer_output.jsonl`


In [7]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'aviation_rag').exists() and (ROOT.parent / 'aviation_rag').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from aviation_rag.config import Settings
from aviation_rag.io_utils import read_jsonl, write_jsonl
from aviation_rag.phase2_san_contract_adapter import Phase2SanContractAdapter
from aviation_rag.phase3_hoang_grounded_qa import Phase3HoangGroundedQA
from aviation_rag.schemas import InputAgentOutput

settings = Settings()
print('Phase 1 artifact:', settings.phase1_output_path)
print('Phase 2 artifact:', settings.phase2_output_path)
print('Phase 3 artifact:', settings.phase3_output_path)


Phase 1 artifact: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase1_hoang_intent_routing_output.jsonl
Phase 2 artifact: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase2_san_retrieval_output.jsonl
Phase 3 artifact: C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase3_hoang_grounded_answer_output.jsonl


## Load Phase 1 Artifact


In [8]:
phase1_rows = read_jsonl(settings.phase1_output_path)
if not phase1_rows:
    raise ValueError(f'No rows found in {settings.phase1_output_path}. Run phase 1 notebook first.')
phase1_rows[0]


{'query_id': 'q_incident_001',
 'query_raw': 'engine failure after takeoff with emergency return',
 'query_normalized': 'engine failure after takeoff with emergency return',
 'intent': 'Incident_Report',
 'intent_confidence': 0.6,
 'intent_source': 'heuristic',
 'expanded_queries': ['engine failure after takeoff with emergency return',
  'engine failure after takeoff with emergency return aviation incident report',
  'engine failure after takeoff with emergency return event narrative',
  'engine failure after takeoff with emergency return safety occurrence summary'],
 'rewritten_query': 'aviation incident narrative lookup for: engine failure after takeoff with emergency return',
 'retrieval_plan': {'strategy': 'semantic',
  'fallback_strategy': 'hybrid',
  'top_k': 10,
  'filters': {},
  'routing_reason': 'Narrative incident queries benefit from semantic similarity over safety reports.'},
 'created_at': '2026-06-05T17:07:35.755118'}

## Resolve San Phase 2 Contract

Neu San chua co output that, adapter se dung local retrieval hoac generated mock de demo full flow.


In [9]:
phase2_adapter = Phase2SanContractAdapter(settings)
phase2_rows = []
for row in phase1_rows:
    phase1_output = InputAgentOutput.model_validate(row)
    phase2_output = phase2_adapter.resolve_output(phase1_output)
    phase2_rows.append(phase2_output)

write_jsonl(settings.phase2_output_path, phase2_rows)
[
    {
        'query_id': row.query_id,
        'predicted_intent': row.predicted_intent,
        'adapter_mode': row.retrieval_diagnostics.get('adapter_mode'),
        'topk_docs': len(row.topk_docs),
    }
    for row in phase2_rows
]


[{'query_id': 'q_incident_001',
  'predicted_intent': 'Incident_Report',
  'adapter_mode': 'generated_mock',
  'topk_docs': 2},
 {'query_id': 'q_procedure_001',
  'predicted_intent': 'Technical_Procedure',
  'adapter_mode': 'generated_mock',
  'topk_docs': 2},
 {'query_id': 'q_metadata_001',
  'predicted_intent': 'Metadata_Query',
  'adapter_mode': 'generated_mock',
  'topk_docs': 2},
 {'query_id': 'q_factoid_001',
  'predicted_intent': 'Factoid',
  'adapter_mode': 'generated_mock',
  'topk_docs': 2}]

## Generate Hoang Phase 3 Grounded Output


In [10]:
phase3 = Phase3HoangGroundedQA(settings)
phase1_lookup = {row['query_id']: row for row in phase1_rows}
phase3_rows = []
for phase2_output in phase2_rows:
    query_raw = phase1_lookup[phase2_output.query_id]['query_raw']
    phase3_rows.append(
        phase3.generate(
            question=query_raw,
            middle_output=phase2_output,
            allow_fallback=True,
        )
    )

[
    {
        'query_id': row.query_id,
        'answer_preview': row.answer[:160],
        'hallucination_risk': row.hallucination_risk,
    }
    for row in phase3_rows
]


[{'query_id': 'q_incident_001',
  'answer_preview': "OpenAI key or network is unavailable, so this is Hoang phase 3 grounded fallback output. Most relevant evidence for 'engine failure after takeoff with emergency",
  'hallucination_risk': 0.24528301886792447},
 {'query_id': 'q_procedure_001',
  'answer_preview': "OpenAI key or network is unavailable, so this is Hoang phase 3 grounded fallback output. Most relevant evidence for 'den bao ENG OIL PRESS sang thi lam gi?': Qu",
  'hallucination_risk': 0.2678571428571429},
 {'query_id': 'q_metadata_001',
  'answer_preview': "OpenAI key or network is unavailable, so this is Hoang phase 3 grounded fallback output. Most relevant evidence for 'crosswind turbulence during final approach ",
  'hallucination_risk': 0.30000000000000004},
 {'query_id': 'q_factoid_001',
  'answer_preview': "OpenAI key or network is unavailable, so this is Hoang phase 3 grounded fallback output. Most relevant evidence for 'what is the meaning of MEL in aviation?': Q",

## Export Phase 3 Artifact


In [11]:
write_jsonl(settings.phase3_output_path, phase3_rows)
required = ['query_id', 'answer', 'citations', 'hallucination_risk', 'grounding_report', 'created_at']
for row in phase3_rows:
    payload = row.model_dump(mode='json')
    missing = [key for key in required if key not in payload]
    if missing:
        raise ValueError(f'Missing keys in {payload.get("query_id")}: {missing}')
print(f'Wrote {len(phase3_rows)} rows to {settings.phase3_output_path}')
print('Phase 3 contract validation passed.')


Wrote 4 rows to C:\Users\DELL\Desktop\Vinh Hoang\Master Program\Xử lý ngôn ngữ tự nhiên\Project\artifacts\phase3_hoang_grounded_answer_output.jsonl
Phase 3 contract validation passed.


## Phase 3D - Groundedness and Answer Quality Metrics

Phase 3 should be evaluated as a grounded answer generator, not only as a text generator. This section summarizes answer evidence support using available runtime fields.

Practical metrics:

- Citation coverage: percentage of answers with at least one citation.
- Average citation count: how many retrieved documents support each answer.
- Average hallucination-risk proxy: lower is better.
- Empty-answer rate: percentage of answers with no generated answer text.


In [12]:
import json
from statistics import mean


def as_dict(row):
    if hasattr(row, "model_dump"):
        return row.model_dump()
    if isinstance(row, dict):
        return row
    raise TypeError(f"Unsupported phase3 row type: {type(row)!r}")

quality_rows = []
for raw_row in phase3_rows:
    row = as_dict(raw_row)
    answer = str(row.get('answer', ''))
    citations = row.get('citations', []) or []
    risk = float(row.get('hallucination_risk', 0.0) or 0.0)
    quality_rows.append({
        'query_id': row.get('query_id'),
        'answer_chars': len(answer),
        'citation_count': len(citations),
        'has_citation': bool(citations),
        'hallucination_risk': round(risk, 4),
        'empty_answer': len(answer.strip()) == 0,
    })

summary = {
    'sample_size': len(quality_rows),
    'citation_coverage': round(sum(r['has_citation'] for r in quality_rows) / max(1, len(quality_rows)), 4),
    'avg_citation_count': round(mean(r['citation_count'] for r in quality_rows), 4) if quality_rows else 0.0,
    'avg_hallucination_risk': round(mean(r['hallucination_risk'] for r in quality_rows), 4) if quality_rows else 0.0,
    'empty_answer_rate': round(sum(r['empty_answer'] for r in quality_rows) / max(1, len(quality_rows)), 4),
}

print(json.dumps({'summary': summary, 'rows': quality_rows}, indent=2, ensure_ascii=False))


{
  "summary": {
    "sample_size": 4,
    "citation_coverage": 1.0,
    "avg_citation_count": 2,
    "avg_hallucination_risk": 0.2725,
    "empty_answer_rate": 0.0
  },
  "rows": [
    {
      "query_id": "q_incident_001",
      "answer_chars": 635,
      "citation_count": 2,
      "has_citation": true,
      "hallucination_risk": 0.2453,
      "empty_answer": false
    },
    {
      "query_id": "q_procedure_001",
      "answer_chars": 601,
      "citation_count": 2,
      "has_citation": true,
      "hallucination_risk": 0.2679,
      "empty_answer": false
    },
    {
      "query_id": "q_metadata_001",
      "answer_chars": 639,
      "citation_count": 2,
      "has_citation": true,
      "hallucination_risk": 0.3,
      "empty_answer": false
    },
    {
      "query_id": "q_factoid_001",
      "answer_chars": 549,
      "citation_count": 2,
      "has_citation": true,
      "hallucination_risk": 0.2766,
      "empty_answer": false
    }
  ]
}
